# 12.3 Multi-Head Attention과 Positional Encoding — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter12_3_multihead_positional.ipynb)

책 본문: [12.3 Multi-Head Attention과 Positional Encoding](https://smhanlab.com/book-ml/kor/ml1/chapter12/3.html)

이 노트북은 본문의 멀티헤드 어텐션과 positional encoding의 핵심 수치를 `numpy`로
구현해 확인합니다. ① 서로 다른 "관점"의 두 헤드가 *미러* 가중치를 내는지,
② 3단어 예에서 *패턴이 다른* 3개 헤드(자기/위치/균일)를 히트맵으로,
③ PE의 "시계" 파동과 \(d_{model}=4\) 테이블을, ④ PE가 없으면 *순서를
구분 못 하는* 것(퍼뮤테이션-등변)을, ⑤ PE의 회전/차분 성질을, ⑥
causal mask가 "왼쪽만" 보이게 하는 것을 각각 숫자로 확인합니다.


In [1]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

IMG = "/home/smhan/book-ml/kor/src/images"   # (Colab에서는 /tmp로 바꾸면 됨)
print("numpy", np.__version__)


numpy 2.4.6


In [2]:
def softmax(X, axis=-1):
    m = X.max(axis=axis, keepdims=True)
    e = np.exp(X - m)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, d_k, mask=None):
    """12.2절의 scaled dot-product attention. mask가 있으면 원점수에 더함 (causal)."""
    scores = Q @ K.T / np.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    weights = softmax(scores)
    return weights @ V, weights, scores

def PE(t, d):
    """위치 t의 d차원 positional encoding (짝수=sin, 홀수=cos)."""
    pe = np.zeros(d)
    for i in range(d // 2):
        w = 1.0 / 10000.0 ** (2 * i / d)
        pe[2 * i]   = np.sin(t * w)
        pe[2 * i + 1] = np.cos(t * w)
    return pe
print("helpers ready: attention(Q,K,V,d_k,mask), PE(t,d)")


helpers ready: attention(Q,K,V,d_k,mask), PE(t,d)


## 1. 같은 입력, 서로 다른 "관점"의 두 헤드 — 미러 확인

본문 "손으로 한 번"의 2단어 예: \(x_1=(1,0), x_2=(0,1)\), \(Q=V=X\).
헤드 A는 \(W_K=I\) (자기-주입), 헤드 B는 \(W_K\)가 두 차원을 바꾸는 행렬
\(\begin{pmatrix}0&1\\1&0\end{pmatrix}\) — *Key*만 달라
"이웃 우선" 헤드가 됩니다. 두 가중치 행렬이 **완전한 미러**인지, 그리고
*더하면* 소거되고 *이어붙이면* 보존되는지 확인합니다.


In [3]:
X    = np.array([[1.0, 0.0], [0.0, 1.0]])          # x1=(1,0), x2=(0,1)
swap = np.array([[0.0, 1.0], [1.0, 0.0]])           # 헤드 B의 W_K (두 차원 교환)

oA, wA, _ = attention(X, X,        X, d_k=2)        # 헤드 A: W_K = I
oB, wB, _ = attention(X, X @ swap, X, d_k=2)        # 헤드 B: W_K = swap

print("헤드 A (자기) 가중치:\n", np.round(wA, 3))
print("헤드 B (이웃) 가중치:\n", np.round(wB, 3))
print("미러 확인  (A == B의 열 뒤집기):", np.allclose(wA, wB[:, ::-1]))
print()
print("헤드 A 단어1 출력:", np.round(oA[0], 3), "  헤드 B 단어1 출력:", np.round(oB[0], 3))
print()
# "합치기" -> 소거
s1, s2 = oA[0] + oB[0], oA[1] + oB[1]
print("두 헤드 *더하면*  단어1 =", np.round(s1, 3), " 단어2 =", np.round(s2, 3),
      "  -> 동일?", bool(np.allclose(s1, s2)))
# 이어붙임 (W_O = I_4)
c1 = np.concatenate([oA[0], oB[0]])
c2 = np.concatenate([oA[1], oB[1]])
print("이어붙임(4차원)  단어1 =", np.round(c1, 3))
print("이어붙임(4차원)  단어2 =", np.round(c2, 3))


헤드 A (자기) 가중치:
 [[0.67 0.33]
 [0.33 0.67]]
헤드 B (이웃) 가중치:
 [[0.33 0.67]
 [0.67 0.33]]
미러 확인  (A == B의 열 뒤집기): True

헤드 A 단어1 출력: [0.67 0.33]   헤드 B 단어1 출력: [0.33 0.67]

두 헤드 *더하면*  단어1 = [1. 1.]  단어2 = [1. 1.]   -> 동일? True
이어붙임(4차원)  단어1 = [0.67 0.33 0.33 0.67]
이어붙임(4차원)  단어2 = [0.33 0.67 0.67 0.33]


## 2. 3단어 [개, 고양이, 쫓는다]에 *패턴이 다른* 3개 헤드

각 임베딩은 2차원·노름 3으로 두고, (a) 자기-우위(Q=K=V), (b) 위치-유사
(Key를 1위치 순환), (c) 역할-없음(모든 Key=(1,1)) 세 헤드의 어텐션 가중치를
히트맵으로 그리면 **같은 입력에 서로 다른 "관점"**이 공존하는 모습을
확인할 수 있습니다. 이 SVG가 본문 `ch12_3_multihead_weights.svg`입니다.


In [4]:
words = ["개", "고양이", "쫓는다"]
E = np.array([[3.0, 0.0], [2.4, -1.8], [0.0, -3.0]])   # 각 노름 3
print("임베딩 노름:", np.round(np.linalg.norm(E, axis=1), 4))

Wa = attention(E, E,                  E, d_k=2)[1]  # (a) 자기-우위
Wb = attention(E, np.roll(E, 1, 0),   E, d_k=2)[1]  # (b) 위치-유사 (Key 순환)
Wc = attention(E, np.tile([1.,1.],(3,1)), E, d_k=2)[1]  # (c) 역할 없음

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2))
for ax, W, title in [(axes[0], Wa, "(a) 자기-우위 (Q=K=V)"),
                     (axes[1], Wb, "(b) 위치-유사 (Key 순환)"),
                     (axes[2], Wc, "(c) 역할 없음 (균일)")]:
    im = ax.imshow(W, cmap="viridis")
    ax.set_xticks(range(3)); ax.set_xticklabels(words, fontsize=9)
    ax.set_yticks(range(3)); ax.set_yticklabels(words, fontsize=9)
    ax.set_xlabel("Key (주의의 대상)", fontsize=9); ax.set_ylabel("Query", fontsize=9)
    ax.set_title(title, fontsize=10)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{W[i][j]:.2f}", ha="center", va="center",
                    color="white" if W[i][j] > 0.4 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, label="가중치", fraction=0.046)
fig.suptitle("3개 헤드의 서로 다른 어텐션 패턴 (같은 입력)", fontsize=12)
fig.tight_layout()
fig.savefig(f"{IMG}/ch12_3_multihead_weights.svg")
plt.show()
print("저장됨:", f"{IMG}/ch12_3_multihead_weights.svg")
print("(a) 대각선(자기):", np.round(np.diag(Wa), 3))
print("(b) 다음-단어 대각:", np.round([Wb[0,1], Wb[1,2], Wb[2,0]], 3))
print("(c) 균일:", np.round(Wc[0], 3))


임베딩 노름: [3. 3. 3.]
저장됨: /home/smhan/book-ml/kor/src/images/ch12_3_multihead_weights.svg
(a) 대각선(자기): [0.78  0.736 0.926]
(b) 다음-단어 대각: [0.78  0.736 0.926]
(c) 균일: [0.333 0.333 0.333]


## 3. PE 테이블 (d=4)과 "시계" 파동

위치 \(t\)의 PE를 \(d_{model}=4\)로 계산해 본문의 테이블을 재현하고,
\(d_{model}=512\)에서 가장 빠른 1번 시계(주기 \(2\pi\approx6.3\))와
가장 느린 256번 시계(주기 약 60,000)의 파동을, 그리고 (b)에서
\(d=4\)의 4개 차원(2개 시계)을 그리면 "빠른 시계는 이웃을, 느린
시계는 대략 어디쯤을" 구분하는 구조가 보입니다. 이 SVG가 본문
`ch12_3_positional_encoding.svg`입니다.


In [5]:
print("d_model=4 PE 테이블 (본문과 대조):")
for t in (0, 1, 2, 10):
    print(f"  t={t:2d}: {np.round(PE(t, 4), 4)}")
print()

t = np.linspace(0, 30, 800)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# (a) d=512: 가장 빠른 시계 vs 가장 느린 시계
ax = axes[0]
w_fast = 1.0 / 10000.0 ** (0 / 256)    # = 1
w_slow = 1.0 / 10000.0 ** (255 / 256)  # ~ 0.000104
ax.plot(t, np.sin(t * w_fast), color="#d95f02", lw=2.2, label=f"1번 시계 sin (ω=1, 주기 2π≈6.3)")
ax.plot(t, np.cos(t * w_fast), color="#d95f02", lw=1.0, ls="--", alpha=0.55)
ax.plot(t, np.sin(t * w_slow), color="#1a9641", lw=2.2, label=f"256번 시계 sin (ω≈{w_slow:.5f}, 주기≈60,000)")
ax.plot(t, np.cos(t * w_slow), color="#1a9641", lw=1.0, ls="--", alpha=0.55)
ax.set_xlabel("위치 t"); ax.set_ylabel("PE 값")
ax.set_title("(a) d=512: 빠른 시계 vs 느린 시계"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# (b) d=4: 4개 차원 (2개 시계)
ax = axes[1]
cols = ["#a50f15", "#d95f02", "#1a9641", "#225ea8"]
for dim in range(4):
    ax.plot(t, np.array([PE(tt, 4)[dim] for tt in t]), color=cols[dim], lw=1.6,
            label=f"dim {dim} ({'sin' if dim % 2 == 0 else 'cos'})")
ax.set_xlabel("위치 t"); ax.set_ylabel("PE 값")
ax.set_title("(b) d=4: 4개 차원 (2개 시계)"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(f"{IMG}/ch12_3_positional_encoding.svg")
plt.show()
print("저장됨:", f"{IMG}/ch12_3_positional_encoding.svg")
print("256번 시계 주기 =", round(2 * np.pi / w_slow, 0))


d_model=4 PE 테이블 (본문과 대조):
  t= 0: [0. 1. 0. 1.]
  t= 1: [0.8415 0.5403 0.01   1.    ]
  t= 2: [ 0.9093 -0.4161  0.02    0.9998]
  t=10: [-0.544  -0.8391  0.0998  0.995 ]



저장됨: /home/smhan/book-ml/kor/src/images/ch12_3_positional_encoding.svg
256번 시계 주기 = 60611.0


## 4. PE 없으면 *순서를 구분 못 한다* — 퍼뮤테이션-등변

본문 "증명" 섹션의 3단어 예: \([개, 고양이, 쫓는다]\) (임베딩 \((1,0),(0,1),(2,0)\))
와 앞 두 단어를 바꾼 \([고양이, 개, 쫓는다]\). PE 없이 \(Q=K=V\)로
self-attention하면 가중치 행렬이 \(W_B = P\,W_A\,P^T\)로 *정확히*
퍼뮤테이션되고, 같은 단어("고양이")의 출력이 위치와 무관하게 *같아*
"누가 누구를 쫓는지"가 사라집니다. 반대로 PE를 **더하면** 같은 단어라도
위치에 따라 가중치가 달라집니다.


In [6]:
E = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, 0.0]])   # 개, 고양이, 쫓는다
P = np.array([[0, 1, 0], [1, 0, 0], [0, 0, 1]], float)  # 앞 두 단어 교환

oA, wA, _ = attention(E,   E,   E,   d_k=2)   # 문장 A
oB, wB, _ = attention(P@E, P@E, P@E, d_k=2)   # 문장 B (재배열)
print("W_B == P W_A P^T :", bool(np.allclose(wB, P @ wA @ P.T)))
print("문장 A '고양이'(2번째) 출력:", np.round(oA[1], 3))
print("문장 B '고양이'(1번째) 출력:", np.round(oB[0], 3))
print("-> 위치와 무관하게 *정확히 동일* (누가 누구를 쫓는지 소실)\n")

# PE를 더하면 달라진다
Ep = E + np.stack([PE(t, 2) for t in (0, 1, 2)])
oAp, wp, _ = attention(Ep, Ep, Ep, d_k=2)
print("동사 '쫓는다' 가중치 행, PE *없*음:", np.round(wA[2], 3))
print("동사 '쫓는다' 가중치 행, PE *있*음:", np.round(wp[2], 3))
print("-> PE가 있으면 같은 단어라도 *위치에 따라* 어텐션이 달라진다")


W_B == P W_A P^T : True
문장 A '고양이'(2번째) 출력: [0.745 0.503]
문장 B '고양이'(1번째) 출력: [0.745 0.503]
-> 위치와 무관하게 *정확히 동일* (누가 누구를 쫓는지 소실)

동사 '쫓는다' 가중치 행, PE *없*음: [0.187 0.045 0.768]
동사 '쫓는다' 가중치 행, PE *있*음: [0.013 0.008 0.979]
-> PE가 있으면 같은 단어라도 *위치에 따라* 어텐션이 달라진다


## 5. PE의 회전 성질과 차분 성질

한 시계 쌍은 각도 \(t\omega\)의 "시계 바늘"이므로, 위치를 \(k\)만큼
앞서는 것은 2×2 회전행렬 \(R(\omega)^k\)를 적용한 것과 *같다*:
\(PE_{(t+k)} = R^k PE_{(t)}\). (b)에 수반되는 *차분* 성질
\(PE_{(t+k)} - PE_{(t)} = (R^k - I)\,PE_{(t)}\)는 절대 위치 \(t\)에
무관하게 *거리* \(k\)만 담는 패턴을 만든다 — 어텐션의 *선형* 도구로
상대 위치를 읽을 수 있는 근거다.


In [7]:
# (a) 회전: d=4의 1번 시계 (ω=1), 1위치 이동 = 1라디안 회전
R1 = np.array([[np.cos(1),  np.sin(1)],
               [-np.sin(1), np.cos(1)]])
print("PE(3) 앞 2성분 :", np.round(PE(3, 4)[:2], 4))
print("R(1)·PE(3) 앞 2성분 :", np.round(R1 @ PE(3, 4)[:2], 4))
print("PE(4) 앞 2성분 :", np.round(PE(4, 4)[:2], 4))
print("회전 성질 성립:", bool(np.allclose(R1 @ PE(3, 4)[:2], PE(4, 4)[:2])))
print()

# (b) 차분: t=3, k=2 (d=4)에서 PE(5)-PE(3) == (R^2 - I) PE(3)
def PE_rot(k, d):
    R = np.eye(d)
    for i in range(d // 2):
        w = 1.0 / 10000.0 ** (2 * i / d)
        R[2*i:2*i+2, 2*i:2*i+2] = np.array([[np.cos(k*w),  np.sin(k*w)],
                                            [-np.sin(k*w), np.cos(k*w)]])
    return R
t, k = 3, 2
Rk = PE_rot(k, 4)
lhs = PE(t + k, 4) - PE(t, 4)
rhs = (Rk - np.eye(4)) @ PE(t, 4)
print("PE(5) - PE(3)      =", np.round(lhs, 4))
print("(R^2 - I)·PE(3)    =", np.round(rhs, 4))
print("차분 성질 성립:", bool(np.allclose(lhs, rhs)))


PE(3) 앞 2성분 : [ 0.1411 -0.99  ]
R(1)·PE(3) 앞 2성분 : [-0.7568 -0.6536]
PE(4) 앞 2성분 : [-0.7568 -0.6536]
회전 성질 성립: True

PE(5) - PE(3)      = [-1.1000e+00  1.2737e+00  2.0000e-02 -8.0000e-04]
(R^2 - I)·PE(3)    = [-1.1000e+00  1.2737e+00  2.0000e-02 -8.0000e-04]
차분 성질 성립: True


## 6. Causal Mask — "왼쪽만" 보기

디코더가 \(t\)번째 단어를 만드는 순간, 그 *이후*("미래")는 보아서는
안 됩니다. softmax *이전* 원점수의 상삼각("미래" 위치)에
\(-10^9\)을 *더하면* 해당 위치의 가중치가 정확히 0이 됩니다.
3단어 예에서 마스크 *없는* 것과 *causal*인 것을 비교합니다.


In [8]:
E = np.array([[1.0, 0.0], [0.0, 1.0], [2.0, 0.0]])   # 개, 고양이, 쫓는다
words = ["개", "고양이", "쫓는다"]
_, w_nomask, _ = attention(E, E, E, d_k=2)
mask = np.triu(np.full((3, 3), -1e9), k=1)           # 상삼각 = "미래"
_, w_mask, _ = attention(E, E, E, d_k=2, mask=mask)

print("마스크 *없는* 가중치:\n", np.round(w_nomask, 3))
print("\ncausal(마스크 *있*은) 가중치:\n", np.round(w_mask, 3))
print("\n첫 단어 '개'는 이제 *자기 자신*만 본다:", np.round(w_mask[0], 3))
print("행 3(쫓는다)은 오른쪽이 없어 마스크 전/후 동일:",
      bool(np.allclose(w_nomask[2], w_mask[2])))


마스크 *없는* 가중치:
 [[0.284 0.14  0.576]
 [0.248 0.503 0.248]
 [0.187 0.045 0.768]]

causal(마스크 *있*은) 가중치:
 [[1.    0.    0.   ]
 [0.33  0.67  0.   ]
 [0.187 0.045 0.768]]

첫 단어 '개'는 이제 *자기 자신*만 본다: [1. 0. 0.]
행 3(쫓는다)은 오른쪽이 없어 마스크 전/후 동일: True


## 7. 정리

| 실험 | 관측 | 본문의 어떤 주장을 확인하나 |
|---|---|---|
| 2단어 두 헤드 | 가중치 미러, 더하면 [1,1] 소거, 이어붙임은 보존 | 멀티헤드 = *관점*의 병렬, concatenate가 소거 방지 |
| 3단어 3헤드 히트맵 | (a) 대각선 / (b) 순환 대각 / (c) 균일 0.333 | 헤드마다 *서로 다른 패턴*이 공존 |
| PE 테이블 + 시계 | d=4 테이블 재현, d=512 빠름(6.3) vs 느림(60,000) | PE = 주파수가 다른 "시계"들의 합 |
| PE 없는 재배열 | \(W_B = P W_A P^T\), "고양이" 출력 위치 무관 [0.745, 0.503] | 어텐션은 *퍼뮤테이션-등변* → PE가 필수 |
| PE 회전/차분 | \(R^k PE_{(t)} = PE_{(t+k)}\), \((R^k-I)PE_{(t)}\) = 차분 | 상대 위치가 *선형 도구*로 읽힌다 |
| causal mask | 상삼각 −10⁹ → 미래 가중치 0, 첫 단어 [1,0,0] | 학습 병렬성 + 생성 순차성의 통일 |
